# Embedding Model using `sentence_transformer`

In this notebook, we explore the way that text with multiple words/tokens is encoded into an embedding vector using the popular `sentence_transformer` library.

We will check:
- OpenAI Embedding
- Open source encoder input embeddings
- Open source encoder output embedding (with context)
- Improved encoder for queries and documents (bi-encoder)

In [1]:
# Define rich theme for better print results
from rich.console import Console
from rich_theme_manager import Theme, ThemeManager
import pathlib

theme_dir = pathlib.Path("themes")
theme_manager = ThemeManager(theme_dir = theme_dir)
dark = theme_manager.get("dark")

# Create a console with dark theme
console = Console(theme = dark)

In [2]:
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

## OpenAI Embedding
A common option is to use the embedding from the same provider as the generation model.

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
first_sentence = "I have no interest in politics"

In [5]:
from openai import OpenAI
client = OpenAI()
response = client.embeddings.create(
    input=first_sentence,
    model="text-embedding-3-small"
)

# console.print(response)

In [6]:
# Create a preview of the response with truncated embedding for better readability
preview = response.model_copy(deep=True)
preview.data[0].embedding = f"{preview.data[0].embedding[:3]}...{preview.data[0].embedding[-3:]}"
console.print(preview)


CreateEmbeddingResponse(
    data=[
        Embedding(
            embedding='[-0.03131103515625, -0.00225830078125, -0.0244598388671875]...[0.012664794921875, 
0.0153045654296875, 0.020843505859375]',
            index=0,
            object='embedding'
        )
    ],
    model='text-embedding-3-small',
    object='list',
    usage=Usage(prompt_tokens=6, total_tokens=6)
)

## Open source encoder = input embeddings
We will start with popular encoders from the `sentence_transformers` library.

It will allow us to explore its architecture and flow, and later on to optimize it to our use-case

In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

### Model Tokenizer
We will use the default tokenizer of the model.

Every word or sub-word is converted into a token with a constant ID.

E.g., in the following two sentences: the word `interest` is tokenized to the same ID.

In [8]:
first_sentence = "I have no interest in politics"
second_sentence = "The bank's interest rate rises"

In [9]:
tokenized_first_sentence = model.tokenize([first_sentence])
console.rule(f"{first_sentence}")
console.print(tokenized_first_sentence)

───────────────────────────────────────── I have no interest in politics ──────────────────────────────────────────

{
    'input_ids': tensor([[ 101, 1045, 2031, 2053, 3037, 1999, 4331,  102]]),
    'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])
}

In [10]:
tokenized_second_sentence = model.tokenize([second_sentence])
console.rule(f"{second_sentence}")
console.print(tokenized_second_sentence)

───────────────────────────────────────── The bank's interest rate rises ──────────────────────────────────────────

{
    'input_ids': tensor([[ 101, 1996, 2924, 1005, 1055, 3037, 3446, 9466,  102]]),
    'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
}

The word `interest` in both sentences is tokenised to `3037`

The token ID can be used to convert it back into readable text.

In [11]:
sentence_tokens = (
    model
    .tokenizer
    .convert_ids_to_tokens(
        tokenized_first_sentence["input_ids"][0]
    )
)

console.print(sentence_tokens)

['[CLS]', 'i', 'have', 'no', 'interest', 'in', 'politics', '[SEP]']

### Model Vocabulary
We will see how many words the model knows in total and preview few of those.

In [12]:
vocabulary = (
    model
    ._first_module()
    .tokenizer
    .get_vocab()
    .items()
)

console.print("[bold] Vocabulary Size: [/bold]", len(vocabulary))
console.print(dict(list(vocabulary)[:10]))

 Vocabulary Size:  30522

{
    'sapphire': 21965,
    '↓': 1586,
    '##hs': 7898,
    '←': 1583,
    'wind': 3612,
    '##hire': 20908,
    'passes': 5235,
    'arden': 26225,
    'ached': 15043,
    'kia': 27005
}

We will search for the token for interest and see its neighbors.

In [13]:
sorted_vocabulary = sorted(
    vocabulary,
    key=lambda x: x[1] # Uses the value of the dictionary entry
)

sorted_tokens = [token for token, _ in sorted_vocabulary]

console.print(sorted_vocabulary[:10])
console.print("total length of sorted vocabulary: ", len(sorted_vocabulary))
console.print(sorted_tokens[:10])
console.print("total length of sorted tokens: ", len(sorted_tokens))


[
    ('[PAD]', 0),
    ('[unused0]', 1),
    ('[unused1]', 2),
    ('[unused2]', 3),
    ('[unused3]', 4),
    ('[unused4]', 5),
    ('[unused5]', 6),
    ('[unused6]', 7),
    ('[unused7]', 8),
    ('[unused8]', 9)
]

total length of sorted vocabulary:  30522

[
    '[PAD]',
    '[unused0]',
    '[unused1]',
    '[unused2]',
    '[unused3]',
    '[unused4]',
    '[unused5]',
    '[unused6]',
    '[unused7]',
    '[unused8]'
]

total length of sorted tokens:  30522

In [14]:
focused_token = 'interest'

# find the index of the 'interest' token
focused_index = sorted_tokens.index(focused_token)

console.print("Index of the word 'interest' was found to be:", focused_index)

Index of the word 'interest' was found to be: 3037

Get 20 tokens nearest to the focused token.

In [15]:
start_index = max(0, focused_index - 10)
end_index = min(len(sorted_tokens), focused_index + 11)
tokens_around_focused_index = sorted_tokens[start_index:end_index]

# console.print("Tokens around 'interest':", tokens_around_focused_index)

from rich.table import Table

table = Table(title=f"Tokens around '{focused_token}':")
table.add_column("id", justify="right", style="cyan", no_wrap=True)
table.add_column("token", style="bright_green")

for i, token in enumerate(tokens_around_focused_index, start=start_index):
    if token == focused_token:
        table.add_row(f"[bold][black on yellow]{i}[/black on yellow][/bold]", f"[bold][black on yellow]{token}[/black on yellow][/bold]")
    else:
        table.add_row(str(i), token)

console.print(table)

     Tokens around     
      'interest':      
┏━━━━━━┳━━━━━━━━━━━━━━┓
┃   id ┃ token        ┃
┡━━━━━━╇━━━━━━━━━━━━━━┩
│ 3027 │ ft           │
│ 3028 │ valley       │
│ 3029 │ organization │
│ 3030 │ stopped      │
│ 3031 │ onto         │
│ 3032 │ countries    │
│ 3033 │ parts        │
│ 3034 │ conference   │
│ 3035 │ queen        │
│ 3036 │ security     │
│ 3037 │ interest     │
│ 3038 │ saying       │
│ 3039 │ allowed      │
│ 3040 │ master       │
│ 3041 │ earlier      │
│ 3042 │ phone        │
│ 3043 │ matter       │
│ 3044 │ smith        │
│ 3045 │ winning      │
│ 3046 │ try          │
│ 3047 │ happened     │
└──────┴──────────────┘

#### The Embedding Transformer Model

Transformer consists of multiple stack modules.

Tokens are an input of the first one.

Let's see that first model.

In [16]:
console.print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True,
'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': 
False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [17]:
first_module = model._first_module()
console.print(first_module.auto_model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (dense): Linear(in_features=384, out_features=1536, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): BertOutput(
          (dense): Linear(in_features=1536, out_features=384, bias=True)
          (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (pooler): BertPooler(
    (dense): Linear(in_features=384, out_features=384, bias=True)
    (activation): Tanh()
  )
)

Get the `embeddings` from the model. 

In [18]:
embeddings = first_module.auto_model.embeddings
console.print(embeddings)

BertEmbeddings(
  (word_embeddings): Embedding(30522, 384, padding_idx=0)
  (position_embeddings): Embedding(512, 384)
  (token_type_embeddings): Embedding(2, 384)
  (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

#### Embedding Model Input Token IDs

We will send the two senecens above to the transformer model. Check the embeddin similarity between the input tokens. 

In [19]:
import torch
device = torch.device("cpu")
# console.print(device)

with torch.no_grad():
    
    # Tokenise both texts
    first_tokens = model.tokenize([first_sentence])
    second_tokens = model.tokenize([second_sentence])
    
    # Get the corresponding embeddings
    first_embeddings = embeddings.word_embeddings(
        first_tokens["input_ids"].to(device)
    )
    
    second_embeddings = embeddings.word_embeddings(
        second_tokens["input_ids"].to(device)
    )

console.print(first_embeddings.shape, second_embeddings.shape)

torch.Size([1, 8, 384])
torch.Size([1, 9, 384])

In [20]:
console.print(first_embeddings)

tensor([[[-0.0176, -0.0076,  0.0471,  ..., -0.0545,  0.0076, -0.0617],
         [-0.0448, -0.0583, -0.0020,  ..., -0.0494, -0.0888, -0.0592],
         [-0.0842, -0.0418, -0.0391,  ...,  0.0415, -0.0840,  0.0829],
         ...,
         [ 0.0099, -0.0220, -0.0515,  ...,  0.0136,  0.0406,  0.1106],
         [ 0.0012, -0.0200,  0.0865,  ..., -0.0331,  0.0299, -0.0233],
         [ 0.0332, -0.0085, -0.0400,  ...,  0.0207, -0.0034, -0.0004]]])

In [21]:
console.print(second_embeddings)

tensor([[[-0.0176, -0.0076,  0.0471,  ..., -0.0545,  0.0076, -0.0617],
         [-0.0042, -0.0197,  0.0089,  ...,  0.0016,  0.0310,  0.1551],
         [ 0.0492,  0.0312, -0.0702,  ...,  0.0487,  0.0661, -0.1310],
         ...,
         [-0.0562, -0.0336, -0.0735,  ..., -0.0053, -0.0379, -0.0688],
         [ 0.0684, -0.0754, -0.0115,  ...,  0.0460,  0.0247,  0.0582],
         [ 0.0332, -0.0085, -0.0400,  ...,  0.0207, -0.0034, -0.0004]]])

In [22]:
from rich.table import Table

table = Table(title = "Embeddings Shape Explanation")

table.add_column("Text", style="cyan", no_wrap=True)
table.add_column("Batch Size", style="magenta")
table.add_column("Tokens Number", style="magenta")
table.add_column("Embedding Dimension", style="magenta")

table.add_row(
    first_sentence,
    str(first_embeddings.shape[0]),
    str(first_embeddings.shape[1]),
    str(first_embeddings.shape[2]),
)
table.add_row(
    second_sentence,
    str(second_embeddings.shape[0]),
    str(second_embeddings.shape[1]),
    str(second_embeddings.shape[2]),
)

console.print(table)

                            Embeddings Shape Explanation                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Text                           ┃ Batch Size ┃ Tokens Number ┃ Embedding Dimension ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ I have no interest in politics │ 1          │ 8             │ 384                 │
│ The bank's interest rate rises │ 1          │ 9             │ 384                 │
└────────────────────────────────┴────────────┴───────────────┴─────────────────────┘

#### Compare the input embedding of the tokens

In [23]:
from sentence_transformers import util
import altair as alt
import pandas as pd

# Calculate cosine similarity
distances = util.cos_sim(
    first_embeddings.squeeze(), 
    second_embeddings.squeeze()
).cpu().numpy()

# Get token labels
x_labels = model.tokenizer.convert_ids_to_tokens(second_tokens["input_ids"][0])
y_labels = model.tokenizer.convert_ids_to_tokens(first_tokens["input_ids"][0])

# Create a DataFrame for Altair
data = pd.DataFrame(
    [(x, y, distances[i, j]) for i, y in enumerate(y_labels) for j, x in enumerate(x_labels)],
    columns=['x', 'y', 'similarity']
)

# Create heatmap using Altair
chart = alt.Chart(data).mark_rect().encode(
    x=alt.X('x:O', title='Second Sentence Tokens', axis=alt.Axis(labelAngle=-45), sort=x_labels),
    y=alt.Y('y:O', title='First Sentence Tokens', sort=y_labels),
    color=alt.Color('similarity:Q', scale=alt.Scale(scheme='yellowgreenblue')),
    tooltip=['x', 'y', alt.Tooltip('similarity:Q', format='.2f')]
).properties(
    width=500,
    height=400,
    title='Input Token Similarity Heatmap'
)

# Add text labels
text = chart.mark_text(baseline='middle').encode(
    text=alt.Text('similarity:Q', format='.2f'),
    color=alt.condition(
        alt.datum.similarity > 0.5,
        alt.value('white'),
        alt.value('black')
    )
)

# Combine chart and text
final_chart = (chart + text).configure_title(fontSize=16)

# Display the chart
final_chart

alt.LayerChart(...)

- `[CLS]` and `[SEP]` tokens are identical in both sentences → similarity = 1.00
- The word `interest` also has similarity = 1.00 in both sentences, even though it means different things — because no context has been applied yet, just a word embedding lookup.
- Other token similarities are small and not important here.

#### Vocabulary Embedding

As we saw there are 30,522 tokens in the vocabulary and each token is embedded with a vector of size 384.

In [24]:
token_embeddings = first_module.auto_model \
    .embeddings \
    .word_embeddings \
    .weight \
    .detach() \
    .cpu() \
    .numpy()
    
console.print(token_embeddings.shape)

(30522, 384)

#### Reduce the Embedding vectors to 2D for visualisation

We will use TSNE library to create a 2D visualisation of the token embeddings, to allow us to see tokens that are close to one another.

This process can take sometime.

In [25]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components = 2, metric = "cosine", random_state = 42)
tsne_embeddings_2d = tsne.fit_transform(token_embeddings)
console.print(tsne_embeddings_2d.shape)

(30522, 2)

#### Token Embedding Visualisation

Once we have the 384 dimension reduced to 2D, we can plot it to explore it.

In [26]:
token_colors = []

for token in sorted_tokens:
    if token[0] == "[" and token[-1] == "]":
        token_colors.append("red")
    elif token.startswith("##"):
        token_colors.append("blue")
    else:
        token_colors.append("green")

In [29]:
import altair as alt
import pandas as pd

# Enable VegaFusion data transformer to handle larger datasets
alt.data_transformers.enable("vegafusion")

# Create a dataframe from data
df = pd.DataFrame({
    'x': tsne_embeddings_2d[:, 0],
    'y': tsne_embeddings_2d[:, 1],
    'token': sorted_tokens,
    'color': token_colors
})

# Create the Altair chart
chart = alt.Chart(df).mark_circle(size = 30).encode(
    x = 'x:Q',
    y='y:Q',
    color = alt.Color('color:N', scale = None),
    tooltip = ['token:N']
).properties(
    width = 600,
    height = 900,
    title = "Token Embeddings"
).interactive()

# Display the chart
chart

alt.Chart(...)

## Open source encoder - output embedding (with context)

Now let's see the token embeddings at the output of the transformer embedding model.

In [30]:
output_embedding = model.encode([first_sentence])
console.print(output_embedding.shape)

(1, 384)

In [31]:
output_token_embeddings = model.encode(
    [first_sentence],
    output_value="token_embeddings"
)

console.print(output_token_embeddings[0].shape)

torch.Size([8, 384])

In [32]:
with torch.no_grad():
    first_tokens = model.tokenize([first_sentence])
    second_tokens = model.tokenize([second_sentence])
    
    first_output_embeddings = model.encode(
        [first_sentence],
        output_value="token_embeddings"
    )
    
    second_output_embeddings = model.encode(
        [second_sentence],
        output_value="token_embeddings"
    )
    
# Calculate cosine similarity
distances = util.cos_sim(
    first_output_embeddings[0],
    second_output_embeddings[0]
)